# Model Persistence and Reproducible Inference

> Save the full preprocessing-and-model pipeline and verify consistent predictions.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Save and load a fitted pipeline with `joblib`.
- Preserve input schema and model metadata.
- Understand serialization risks and reproducibility limits.

A model artifact should include the fitted preprocessing steps, expected input columns, package versions, training data reference, and evaluation summary. Only load artifacts from trusted sources.

In [1]:
from pathlib import Path
import json, tempfile
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42)).fit(X_train,y_train)
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "model.joblib"
    joblib.dump(pipe, path)
    loaded = joblib.load(path)
    original = pipe.predict_proba(X_test.iloc[:5])
    restored = loaded.predict_proba(X_test.iloc[:5])
    print("Max prediction difference:", np.max(np.abs(original-restored)))
    print("Scikit-learn version:", sklearn.__version__)

Max prediction difference: 0.0
Scikit-learn version: 1.8.0


## Record the model contract
A production client needs to know required feature names, types, allowed ranges, output meaning, and version. A model file alone is not an API contract.

In [2]:
model_card = {
    "model_name": "breast-cancer-logistic-regression-demo",
    "target_semantics": "Dataset target encoding from sklearn.datasets.load_breast_cancer",
    "features": list(X.columns),
    "training_seed": 42,
    "scikit_learn_version": sklearn.__version__,
    "metrics": "Replace this field with held-out metrics for a real project",
    "known_limitations": ["Teaching dataset", "Not validated for clinical use"],
}
print(json.dumps(model_card, indent=2)[:1200])

{
  "model_name": "breast-cancer-logistic-regression-demo",
  "target_semantics": "Dataset target encoding from sklearn.datasets.load_breast_cancer",
  "features": [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
    "mean compactness",
    "mean concavity",
    "mean concave points",
    "mean symmetry",
    "mean fractal dimension",
    "radius error",
    "texture error",
    "perimeter error",
    "area error",
    "smoothness error",
    "compactness error",
    "concavity error",
    "concave points error",
    "symmetry error",
    "fractal dimension error",
    "worst radius",
    "worst texture",
    "worst perimeter",
    "worst area",
    "worst smoothness",
    "worst compactness",
    "worst concavity",
    "worst concave points",
    "worst symmetry",
    "worst fractal dimension"
  ],
  "training_seed": 42,
  "scikit_learn_version": "1.8.0",
  "metrics": "Replace this field with held-out metrics for a real project",
 

## Serialization and safety notes
- `joblib` and pickle-based formats can execute code during loading; never load untrusted artifacts.
- Package-version changes can break compatibility or change behavior.
- A random seed does not guarantee bit-for-bit reproducibility across every hardware/library version.
- Use a model registry and artifact checksums for collaborative deployments.

## Practice
Add the training date and a dataset identifier to the model card. Explain why a notebook output alone is not sufficient reproducibility documentation.